# Build 03-05 · Re-evaluation — operating points per axis on the OOT split

**Kernel: the analysis `.venv`** (`python3`) — reads parquet + CSV only, loads no model.

Compares the retrained axes (and the baseline, when its `scores` kind exists) on the version's
**OOT split**, on **garage rows only** (`decision = 0` — the verified slice; scrapped rows carry
forced labels and cannot score precision). Decisions use the **STRICT rule `score > τ`**
(`score == τ` garages), matching `threshold.apply`.

τ sources per model:
- **grid** — the production rule's own value(s) (`config.DECISION_RULES`): the "what if the
  production cutoff were kept" reading. A retrained model's score scale is its own, so these
  rows are reference, not the comparable operating point.
- **tuned** — `threshold.tune` on **train-split** garage rows: lowest cutoff with precision ≥
  target, the precision-floor mode of the company's own `select_best_threshold` (adopted into
  `tune()` 2026-09-02). This is the comparable operating point.

```
mitigation/inputs/corrector_targets_<v>_<oot> (03_01)   reeval/<v>_mitigated_scores_<oot>_<tag>
   ──▶  §3 metrics per (model, τ): AUC · ψ · precision · recall · TN/FP/FN/TP · scrap rate
          └▶ reeval/<v>_decisions_<oot>_<model>_tau<τ>.parquet   (one file per τ — 여러 갈래)
          └▶ reeval/<v>_reweight_metrics_<oot>.csv
   ──▶  §4 PR curves (figstyle house style) ──▶ figures/
```

In [ ]:
# §0 — setup (analysis .venv kernel)
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.metrics import precision_recall_curve, roc_auc_score

ROOT = Path.cwd()
while not (ROOT / "src" / "config.py").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))

import config
import figstyle
import threshold
from detector.algorithm.residual_peak import peak0, residual

figstyle.apply()
pd.set_option("display.width", 160)
print("ROOT =", ROOT)

In [ ]:
# §1 — RUN_SPEC
VERSION = "v3"        # "v2" works too once 03_01 built its corrector_targets
TRAIN_SPLIT = "train"                             # where the tau tuning reads
EVAL_SPLIT = config.OOT_SPLIT[VERSION]            # never picked by name — v1/v2 invert "test"
ID_COL = "claim_id"
SCORE_COL = "model_" + VERSION + "_score"
TARGET_PREC = config.TARGET_PRECISION
TUNE_TAU = True

# grid = the production rule's own cutoff value(s); override with an explicit list if needed
_rule = config.DECISION_RULES[VERSION]
if _rule["shape"] == "global":
    TAU_GRID = [float(_rule["threshold"])]
elif _rule["shape"] == "piecewise_global":
    TAU_GRID = sorted({float(r["threshold"]) for r in _rule["regimes"]})
else:
    raise SystemExit("segmented rule (v1) is out of scope here")

REEVAL_DIR = ROOT / "src" / "data" / "real" / "reeval"
print("eval split:", EVAL_SPLIT, "| tau grid:", TAU_GRID)

In [ ]:
# §2 — assemble: corrector_targets (treatment pre-joined by 03_01) + one score column per model
def load_split_frame(split: str) -> pd.DataFrame:
    """corrector_targets of `split` — targets already joined to the recorded treatment (03_01)."""
    p = config.split_path("corrector_targets", VERSION, split)
    assert p.is_file(), f"{p} missing — run 03_01_corrector_inputs first"
    m = pd.read_parquet(p)
    m["date"] = pd.to_datetime(m["date"])
    print(f"{VERSION} {split}: {len(m):,} treated rows (unmatched already dropped in 03_01)")
    return m


def attach_scores(frame: pd.DataFrame, split: str) -> tuple[pd.DataFrame, list[str]]:
    """Merge every available model's scores on `split`: mitigated axes + baseline if exported."""
    models = []
    bpath = config.path("scores", VERSION, split=split)
    if bpath.is_file():
        b = pd.read_parquet(bpath)[[ID_COL, SCORE_COL]].rename(columns={SCORE_COL: "baseline"})
        frame = frame.merge(b, on=ID_COL, how="left")
        models.append("baseline")
    else:
        print(f"  note: no baseline scores at {bpath} (run src/scoring/score_all.py) — skipped")
    prefix = f"{VERSION}_mitigated_scores_{split}_"
    for p in sorted(REEVAL_DIR.glob(prefix + "*.parquet")):
        tag = p.stem[len(prefix):]
        s = pd.read_parquet(p)[[ID_COL, SCORE_COL]].rename(columns={SCORE_COL: tag})
        frame = frame.merge(s, on=ID_COL, how="left")
        models.append(tag)
    return frame, models


ev, MODELS = attach_scores(load_split_frame(EVAL_SPLIT), EVAL_SPLIT)
tr, tr_models = attach_scores(load_split_frame(TRAIN_SPLIT), TRAIN_SPLIT)
assert MODELS, "no score files for the eval split — run 03_04 first"

matched = ev["decision"].notna()
ev_gar = ev.loc[matched & (ev["decision"] == 0)]
tr_gar = tr.loc[tr["decision"].notna() & (tr["decision"] == 0)]
print(f"eval garage rows: {len(ev_gar):,} ({int(ev_gar['observed'].sum()):,} verified TL) | "
      f"unmatched (no treatment): {int((~matched).sum()):,} — excluded from garage metrics")
print("models:", MODELS)

In [ ]:
# §3 — metrics per (model, τ): decisions parquet per τ + one CSV
def metrics_at(y: np.ndarray, s: np.ndarray, tau: float) -> dict:
    """AUC + precision/recall/CM at the STRICT rule score > tau (score == tau garages)."""
    pred = s > tau
    tp = int((pred & (y == 1)).sum()); fp = int((pred & (y == 0)).sum())
    tn = int((~pred & (y == 0)).sum()); fn = int((~pred & (y == 1)).sum())
    return {"AUC": float(roc_auc_score(y, s)),
            "precision": tp / (tp + fp) if tp + fp else float("nan"),
            "recall": tp / (tp + fn) if tp + fn else float("nan"),
            "TN": tn, "FP": fp, "FN": fn, "TP": tp}


def recall_at_precision(y: np.ndarray, s: np.ndarray, target: float) -> float:
    """Max recall over all cutoffs while precision >= target; NaN if unreachable."""
    prec, rec, _ = precision_recall_curve(y, s)
    ok = prec >= target
    return float(rec[ok].max()) if ok.any() else float("nan")


rows = []
for name in MODELS:
    g = ev_gar[[ID_COL, "observed", name]].dropna(subset=[name])
    if len(g) < len(ev_gar):
        print(f"  {name}: {len(ev_gar) - len(g)} eval garage rows unscored — dropped for this model")
    y = g["observed"].astype(int).to_numpy()
    s = g[name].to_numpy(dtype=float)
    psi = peak0(residual(y, s))                       # threshold-free loop signal, per model
    r_tp = recall_at_precision(y, s, TARGET_PREC)

    taus = [(t, "grid") for t in TAU_GRID]
    if TUNE_TAU:
        if name in tr_models:
            tg = tr_gar[["observed", name]].dropna(subset=[name])
            t = threshold.tune(tg["observed"].astype(int).to_numpy(),
                               tg[name].to_numpy(dtype=float), target=TARGET_PREC)
            if t is None:
                print(f"  {name}: no cutoff reaches precision>={TARGET_PREC} on train garage — no tuned tau")
            else:
                taus.append((float(t), "tuned"))
        else:
            print(f"  {name}: no train-split scores — tuned tau skipped")

    all_scored = ev[[ID_COL, name]].dropna(subset=[name])
    s_all = all_scored[name].to_numpy(dtype=float)
    for tau, src in taus:
        dp = REEVAL_DIR / f"{VERSION}_decisions_{EVAL_SPLIT}_{name}_tau{round(float(tau), 6)}.parquet"
        pd.DataFrame({ID_COL: all_scored[ID_COL].to_numpy(), "score": s_all,
                      "decision": (s_all > tau).astype(int)}).to_parquet(dp, index=False)
        rows.append({"model": name, "tau": round(float(tau), 6), "tau_source": src,
                     "n_eval_garage": len(g), **metrics_at(y, s, float(tau)),
                     "psi_loop": round(psi, 3),
                     "recall_at_P" + str(TARGET_PREC): round(r_tp, 4),
                     "scrap_rate_all": round(float((s_all > float(tau)).mean()), 5)})

metrics = pd.DataFrame(rows)
csv_path = REEVAL_DIR / f"{VERSION}_reweight_metrics_{EVAL_SPLIT}.csv"
metrics.to_csv(csv_path, index=False)
print("->", csv_path)
display(metrics.round(4))

In [ ]:
# §4 — PR trade-off on the eval garage slice (threshold swept), house style
fig, ax = plt.subplots(figsize=figstyle.FIG_1)
for name in MODELS:
    g = ev_gar[["observed", name]].dropna(subset=[name])
    y = g["observed"].astype(int).to_numpy()
    prec, rec, _ = precision_recall_curve(y, g[name].to_numpy(dtype=float))
    if name == "baseline":
        ax.plot(rec[:-1], prec[:-1], ls=":", color=figstyle.NEUTRAL, label="baseline")
    else:
        ax.plot(rec[:-1], prec[:-1], label=name)     # colour cycle assigns slots in order
ax.axhline(TARGET_PREC, ls="--", lw=1, color=figstyle.INK)
ax.text(0.01, TARGET_PREC + 0.01, f"{TARGET_PREC} target", fontsize=8, color=figstyle.INK)
ax.set_xlabel("recall"); ax.set_ylabel("precision"); ax.set_ylim(0, 1.02)
ax.set_title(f"{VERSION} reweight axes — PR on {EVAL_SPLIT} garage rows")
ax.legend(fontsize=8)
figstyle.save(fig, f"{VERSION}_0305_reweight_pr_{EVAL_SPLIT}")
plt.show()

## §5 — how to read this (and what it cannot say)

- **Precision is measured on the verified (garage) slice only.** The high-score region is
  almost empty of garage rows (positivity), so a τ that holds precision on the train slice can
  fail out-of-time — 03_02's synthetic study showed exactly that collapse. Read the *ranking*
  of axes and the `recall_at_P` column, not absolute levels.
- **grid vs tuned rows answer different questions**: grid = the production cutoff imposed on a
  rescaled score (reference only); tuned = each model's own comparable operating point, from
  `threshold.tune`'s precision-floor mode — the company `select_best_threshold` rule, adopted
  2026-09-02.
- ψ (`psi_loop`) is threshold-free and reads the loop signal, not deployability — a scheme can
  ease ψ and still fail the precision constraint.
- Rows the deciding log never scored have no treatment: excluded from garage metrics
  (`scrap_rate_all` still uses every scored row).
- More than ~7 axes will exhaust the 8-slot colour cycle in §4 — fold or split the figure
  rather than inventing a 9th colour.